# Resolve a run and preview PubMed queries

A run must persist the resolved pathogen–host pairs before any acquisition begins. This notebook resolves the pinned pilot configuration and builds versioned pair queries; it does not start network acquisition.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pathogen_atlas_v4').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import yaml
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
config = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
taxonomy = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
hosts = __import__('pandas').read_parquet(taxonomy / 'hosts.parquet')
pathogens = __import__('pandas').read_parquet(taxonomy / 'pathogens.parquet')
resolved = resolve_run(config, known_host_ids=set(hosts.host_id.astype(str)), known_pathogen_ids=set(pathogens.pathogen_id.astype(str)))
len(resolved.pairs), resolved.resolved_config_hash

In [ ]:
save_resolved_run(config, resolved, ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json')
pathogen = pathogens.set_index('pathogen_id').loc[resolved.pairs[0][0]]
host = hosts.set_index('host_id').loc[resolved.pairs[0][1]]
query = build_pair_query([pathogen.accepted_name], [host.accepted_name, *host.aliases], strategy_version=config['acquisition']['query_strategy_version'])
query

Inspect query text and hash before wiring it to a PubMed client. Persist every returned PMID, query hash, retrieval status, and sampling decision as pair-scoped literature records.